# A bare-bones RAG implementation  

Rev. 2

## Overview

Retrieval Augmented Generation (RAG) is intended to alleviate some of the most obvious [problems][1] displayed by Large Language Models (LLMs). Examples of such problems are hallucinations, outdated information and lack of domain-specific knowledge. The latter two are due to LLMs being trained on massive amounts of data aquired before some date (the cutoff date) but once training stops no more information enters the LLM, and domain-specific things like a company's intraweb or customer database was (hopefully) never part of the training data. Hallucinations are due to how LLMs work - they generate probable and well formed sentences, but they don't _understand_ anything.

RAG consists of three steps; *retrieval* of context-specific information, *augmenting* (i.e. adding context to) the LLM prompt, and finally letting the LLM *generate* an answer taking that context into account. This is all done without user intervention.

This notebook intends to demystify the steps involved by performing RAG, getting by without the help of tools like [LangChain][2] that obscures what is really going on.

The example in this notebook is (loosely) based on [this blog post][3].

[1]: https://youtu.be/T-D1OfcDW1M?si=nKf8KC93tcsbbAlO
[2]: https://www.langchain.com
[3]: https://vigneshwarar.substack.com/p/hackernews-support-page-using-retrieval

## Prerequisites

The very first step is to make sure all requirements (in terms of python modules) are present on your system. We'll import (and discuss) each module during the course of this tutorial.

[Run](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Running%20Code.html) the cell below ( i.e. select it and press Shift + Enter) to install the listed modules. 

In [ ]:
!pip -q install ollama qdrant-client==1.11.3 

In [ ]:
import ollama

OLLAMA_MODEL = 'qwen3:4b'
OLLAMA_HOST = 'http://10.129.20.4:9090'
client = ollama.Client(host=OLLAMA_HOST)

## Caveman RAG

We'll begin by doing the simplest possible RAG as an example to keep in mind, it really is this simple even if the actual implementation tends to obscure the simplicity.

The first part of RAG-enabled LLM is ... an LLM that we can query.

We need a some textual instructions to wrap our question in to form a _prompt_: 

In [ ]:
base_prompt = """
You are an AI assistant. Your task is to understand the user question, and provide an answer.

Your answers are short and to the point.
Don't make up answers, rely only on facts.
If you have no facts, simply state "I don't know"

"""

def make_prompt(question: str) -> str:
    return base_prompt + f"User question: {question}"
    


With that we can pose a question:

In [ ]:
prompt = make_prompt("What is WARA-ops?")
print(prompt, "\n---")
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream=False,
)
print(reply.response)

Sometimes the answer is 'I don't know', but not always. Try re-running the cell a couple of times to see different responses

Next, construct a minimal "database" with a single piece of fact:  

In [ ]:
database = {
    "What is WARA-ops?": "WARA-ops is an organization funded by Wallenberg."
}

Now we want to intercept the the query and possibly add context to the prompt _before_ it is sent to the LLM:

In [ ]:
def make_augmented_prompt(question: str) -> str:
    fact = database.get(question)
    if fact:
        return base_prompt + f"Fact: {fact}\n\n" + f"User question: {question}"
    else:
        return base_prompt + f"User question: {question}"

In [ ]:
prompt = make_augmented_prompt("What is WARA-ops?")
print(prompt, "\n---")
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream = False,
)
print(reply.response)

Hopefully the LLM now responds using the stated fact (in some form). Re-run the cell a couple of times to see different results.

The downside is of course that our limited database and naive search mechanism only works for one very specific question. 

Let's try another question:

In [ ]:
prompt = make_augmented_prompt("What is the purpose of WARA-ops?")
print(prompt, "\n---")
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream = False,
)
print(reply.response)

Again, the response _ought to be_ 'I don't know'.

This is how RAG works. The rest of this tutorial will mainly focus on 1) how to contruct a database with facts, and 2) how to match those facts against a general question.

## Baseline

We are going to use information about HackerNews to build our database, but as this information is on the internet the LLM has likely encountered it during training.

Before getting into RAG, and what it contributes, let's establish a baseline by querying our LLM about HackerNews without using RAG.

In [ ]:
prompt = make_prompt("What is special about HackerNews?")
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream = False,
)
print(reply.response)

## Gathering facts

In order to demonstrate RAG capabilities, we need some domain specific facts (context), to work with.
Some of the info-pages from HackerNews (legal.html, newsfaq.html, newsguidelines.html, security.html) was downloaded and converted to plain text and put in JSON-files like so<sup>1</sup>:

```json
    {
        "content": "Hacker News Guidelines\n...",
        "url": "https://news.ycombinator.com/newsguidelines.html"
    }
```

and stored in the `data` directory:

```
data/
    legal.json
    newsfaq.json
    newsguidelines.json
    security.json
```

You can examine the contents of the directory by opening the folder in the left sidebar.

---
<p><small>1. The reason for keeping the url is to be able to (manually) track and reference the original document in the response as a post-query operation.</small></p>

---

## Data ingestion and chunking

Next, we have to somehow prepare our data for use with an LLM prompt; _ingest_ it.

We need to store "units" of facts separately and provide a means to looking up facts relevant to the query.
The goal is to make our custom data ready for _semantic querying_ i.e. searching for text whose meaning somehow relates to our question (see e.g. [King – Man + Woman = Queen](https://www.technologyreview.com/2015/09/17/166211/king-man-woman-queen-the-marvelous-mathematics-of-computational-linguistics/))

Basically, there are two steps required; 1) _chunking_, i.e. turning data into smaller pieces, and 2) turning those chunks into _semantic vectors_ of high dimension (a.k.a _embeddings_) that captures the semantics, the meaning, of the corresponding chunk.

Next, we'll look at chunking in some detail, and postpone explanation of semantic vectors/embeddings to later, just keep the concept in mind for now.

### Chunking

Breaking down text to "units" of facts (pieces of continuous text related to a specific subject) is a process known as chunking. Too small chunks doesn't have enough context, but too large chunks may contain unrelated contexts. This is not an exact science, and domain knowledge and understanding of the process helps in guiding the trade-off choices. Think of chapters, sections, paragraph or sentences in the case of chunking a book or essay.

For this example, ordinary sentences will make up the chunks, but e.g. paragraphs might prove to be a good alternative.

For the sake of clarity, I will show how to split the data into sentences (chunks) using plain python. For a real world deployment a natural language processing library like [spaCy](https://spacy.io) could be a better choice, but for this simple demo it would be total overkill and just obscure what is going on.

In [ ]:
#
# Split the input data into sentence-sized chunks
#
import re
import json

chunks = []
index = 0

filenames = ["newsfaq.json", "newsguidelines.json", "security.json", "legal.json"]
# Iterate over the entries in data/ and read each JSON file in turn
for filename in filenames:
    filepath = f"./data/{filename}"
    with open(filepath) as fd:
        data = json.load(fd)

    url = data['url']
    text = data['content']
    # Split the file's text contents into sentences using python regex:
    #   A sequence of characters is deemed a sentence if followed by a
    #   full stop (.), question mark (?), or an exclamation mark (!)
    #   immediately followed by one or more whitespaces.
    sentences = re.split(r"(?<=\.|\?|!)\s+", text)
    # Each sentence make up a chunk, store it with references (url and id)
    for sentence in sentences:
        chunks.append({'id': index, 'text': sentence, 'url': url})
        index += 1

# Write the resulting array to file:
with open('chunks.json', 'w') as fd:
    json.dump(chunks, fd)

Just a sanity check, it should be ~570 chunks, but we'll limit us to visualize the ten first chunks. Flip them open using the triangles to examine the contents.

In [ ]:
import json
from IPython import display

entries = json.loads(json.dumps(chunks[:10], default=dict))
display.JSON(entries, root=f"The ten first chunks of {len(chunks)}")

## Embedding

Embedding is the process of turning data into semantic vectors representing that data in a way that makes it suitable for computers. 

A semantic vector is a sequence of numbers (vector) that somehow captures the meaning (semantic) of the data. Magic.
Think of such a vector the coordinates of a point in _semantic space_, just like a GPS coordinates point to a location on earth.

The point of computing a semantic vector (embedding) is that vectors can be readily _compared_. That two embeddings are "close" means that their corresponding data is also "close" in some sense.

It is worth pointing out that data could be almost anything; text, images, etc. that can be represented as a coordinate in a semantic space ${\mathbb R}^n$, where $n$ is large, typically 512, 768, 1024 or some such. 

If that didn't help in understanding, consider the two-dimensional space ${\mathbb R}^2$ with `redness` on the x-axis and `blueness` on the y-axis as shown in the picture below. The colour purple, which is (sort of) a mix of red and blue, would be somewhere along the diagonal $y \approx x$.

<img src="img/image3.png" alt="Example" style="width: 400px;"/>

Figure. A two-dimensional toy example trying to illustrate embedding of red- and blueness, where purpleness emerges as a combination of the two.

Embedding, i.e. converting a chunk to a semantic vector by a _vectorizer_, can be done in many ways, most often using a trained neural network. That said, the purpose of any vectorizer is simple: construct a set of vectors that represent the semantic information in the best possible way. 

We'll be using the embedding model `mxbai-embed-large`, simply because it is readily available from our server, and Ollama's `embed()`API.

In [ ]:
embedding_model_name = 'mxbai-embed-large'
# Gather just the text sentences from our chunks
sentences = [chunk['text'] for chunk in chunks]
# Vectorize, i.e. create embeddings
print("Embedding, please wait... (this may take a minute or so)")
response = client.embed(model=embedding_model_name, input=sentences)
embeddings = response.embeddings
print(f"Finished embedding {len(embeddings)} sentences, embedding size: {len(embeddings[0])}")

### Feeding a vector store (database)

We now have text sentences, and an embedding and a URL for each, and need to store them somewhere **and** make them searchable. Storing is straightforward, but searching is in practice not so simple.

Searching means to take a query string, _embed_ it using _the same_ embedding model as the chunk sentences, and find the stored sentence whose embedding most closely matches the query's embeeding.

The reason searching is the bottleneck is mostly due to the sheer number of vectors to search in a realistic case (many millions of entries), our 500 entries represent a puny set to search.

The search operation, _similarity search_ is basically a very simple operation:

Given a set of semantic vectors ${\bf x}_i \in \left\{x_1,\ldots,x_n\right\}$ (i.e. in ${\mathbb R}^n$), find the vector(s) most closely matching a _query vector_ ${\bf x}$ by finding

$i = {\textit argmin}_i||{\bf x} - {\bf x}_i||$,

i.e. (the index of) the most similar vector, where $||\cdot||$ is the Euclidean distance (${\textrm L}_2$) in ${\mathbb R}^n$.

To continue the example from above, a search for the (red, blue) vector that most closely matches <img src="img/query.png" alt="query color" width="17" height="17" style="vertical-align:middle">, whose _embedded query vector_ is (0.29, 0.59), yields an answer of (0.30, 0.55) equivalent to <img src="img/answer.png" alt="query color" width="17" height="17" style="vertical-align:middle">.<sup>2</sup>

Here we'll be using Qdrant, an Open Source _vector database_ service hosted by us.

---
<p><small>2. While the above example might look simple and intuitive, be warned that the ${\textrm L}_2$-norm behaves quite differently in ${\mathbb R}^n$ for $n>3$ than our experience from $n=2$ and $n=3$ lead us to believe. For reasons we're not going into here, ${\textrm L}_1$ or ${\textrm L}_\infty$ could be good options, and vector stores offer additional metrics such as _cosine similarity_.</small></p>

---	

## Retrieval

Retrieval is the process of gathering contextual information, given a user query, to augment the LLM prompt in the hope of getting a more accurate answer.

The first step is to encode the plain text query into a semantic vector _using the same model_ as was used to create the embeddings.

The query embedding lets us search the set of semantic vectors for vectors (and thus chunks/sentences) that are semantically close to the query. We'll sanity check our example by requesting the two most relevant (according to the vectorizing model) sentences, using a query we _know_ is part of the set which means it should come up as the top hit in the search.

### Live example: Qdrant (database)

For this example we'll use a [Qdrant](https://qdrant.tech/qdrant-vector-database/), a database service running on the same server as Ollama.

[Documentation for Qdrant client](https://python-client.qdrant.tech).

First, let's set up a _personal_ space for our toy example and upload the embeddings:

In [ ]:
# Create a named _collection_ making up our corner of the database (it is a shared resource)
collection_name = "" # Add a reasonably unique name of your choice here using a label and date, e.g. "name_date" 

Next, let's start communicating with the database:

In [ ]:
QDRANT_HOST = "10.129.20.4"
QDRANT_PORT = 6333

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

vector_length = len(embeddings[0])

# Create a client connecting to the service
qdrant_store = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

# Check if collection (for this toy example) already exist, and remove if so
if qdrant_store.collection_exists(collection_name=collection_name):
   qdrant_store.delete_collection(collection_name=collection_name)

# Create a named collection and set vector dimension and metric (EUCLID means L2 norm)
qdrant_store.create_collection(
    collection_name = collection_name,
    vectors_config = VectorParams(size=vector_length, distance=Distance.EUCLID),
)

# Upload our embeddings, one variant of many

# If ids are _not_ provided, Qdrant Client will replce them with random UUIDs (not good in this case).
# Optional _payload_ not utilized, could in this example be e.g. the URL associated with each embedding
n = len(embeddings)
qdrant_store.upload_collection(
    collection_name = collection_name,
    ids = range(n), # Provide a list of indices for use as id
    vectors = embeddings,
)

First a sanity check (note that qdrant's similarity search is _guided_ by `SearchParams`, see <https://qdrant.tech/documentation/concepts/search/>):

In [ ]:
# Get a query from the stored sentences
query = sentences[5]
print(query)

In [ ]:
from qdrant_client import models

# embed the query
response = client.embed(model=embedding_model_name, input=query)
query_embedding = response.embeddings[0]

# Return the two closest matches
search_result = qdrant_store.query_points(
    collection_name = collection_name,
    search_params = models.SearchParams(hnsw_ef=10, exact=False),
    query = query_embedding,
    limit = 2,
)

for entry in search_result.points:
    print(f"Found id: {entry.id} with score: {entry.score}. Corresponding sentence: '{sentences[entry.id]}'")

Unsurprisingly, the top hit is id 5 with a score of (or very close to) zero meaning identical.

Now we're ready for a more general test:

In [ ]:
query = 'What is special about HackerNews?'
# embed the query
response = client.embed(model=embedding_model_name, input=query)
query_embedding = response.embeddings[0]

In [ ]:
# Number of matches for search to return
k = 10
search_result = qdrant_store.query_points(
    collection_name = collection_name,
    search_params = models.SearchParams(hnsw_ef= 50, exact=False),
    query = query_embedding,
    limit = k,
)

indices = [res.id for res in search_result.points]

# Context for LLM
context = '\n'.join([f'{idx}. {sentences[idx]}' for idx in indices])

In [ ]:
print(context)

## Prompt preparation (Augment)

### Prompt format

The best prompt format for RAG-augmentation is an open question but we'll keep it simple use the following prompt template:

In [ ]:
base_prompt = """
You are an AI assistant. Your task is to understand the user question, and provide an answer using the provided context.

Your answers are short, to the point, and written by an domain expert. Provide references to the context where appropriate.
If the provided context does not contain the answer, simply state, "The provided context does not have the answer."

Context:
{}

User question: {}
"""

In [ ]:
prompt = base_prompt.format(context, query)

Uncomment next line if you want to see what gets fed into the LLM

In [ ]:
# print(prompt)

### Answer Generation

In [ ]:
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream = False,
)
print(reply.response)

While this answer is probably not too far removed from the one obtained without RAG, it is hopefully more focused and contains references to the context provided by the RAG-added context.

In the next part of this tutorial we'll expand these learnings into a chatbot providing useful guidance when writing makefiles.